# PRUEBA DE PIPELINE DE EVALUACIÓN WHISPERX MEDIUM (VENVNOTEBOOK)

In [ ]:
import os
import time
import torch
import pandas as pd
import whisperx
from jiwer import wer
import re
import unicodedata

# =========================
# NORMALIZACIÓN
# =========================
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = texto.encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r"[^\w\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

# =========================
# CONFIG
# =========================
CARPETA_AUDIO = "llamadas"
EXCEL_PATH = "dataset_llamadas.xlsx"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU disponible:", torch.cuda.is_available())

# =========================
# CARGAR EXCEL
# =========================
df_ref = pd.read_excel(EXCEL_PATH)
mapa_textos = dict(zip(df_ref["audio"], df_ref["texto_real"]))

# =========================
# CARGAR MODELO WHISPERX
# =========================
model = whisperx.load_model("medium", device)

# modelo de alineación (CLAVE 🔥)
model_a, metadata = whisperx.load_align_model(
    language_code="es",
    device=device
)

# =========================
# LOOP AUDIOS
# =========================
resultados = []

for archivo in os.listdir(CARPETA_AUDIO):

    if not archivo.endswith(".wav"):
        continue

    audio_path = os.path.join(CARPETA_AUDIO, archivo)
    print(f"\n🎧 Procesando: {archivo}")

    try:
        start = time.time()

        # 🔊 cargar audio
        audio = whisperx.load_audio(audio_path)

        # 🧠 transcripción base
        result = model.transcribe(audio)

        # 🔥 alineación (esto hace la magia de WhisperX)
        result_aligned = whisperx.align(
            result["segments"],
            model_a,
            metadata,
            audio,
            device
        )

        # unir texto final
        texto_pred = " ".join([seg["text"] for seg in result_aligned["segments"]])

        elapsed = time.time() - start

        # texto real
        texto_real = mapa_textos.get(archivo, None)

        if texto_real:
            wer_score = wer(
                normalizar(texto_real),
                normalizar(texto_pred)
            )
        else:
            wer_score = None

        resultados.append({
            "audio": archivo,
            "texto_real": texto_real,
            "texto_pred": texto_pred,
            "wer": wer_score,
            "tiempo_seg": elapsed
        })

        print(f"WER: {wer_score:.3f}" if wer_score else "Sin referencia")
        print(f"Tiempo: {elapsed:.2f}s")

    except Exception as e:
        print(f"❌ Error en {archivo}: {e}")

# =========================
# RESULTADOS
# =========================
df_resultados = pd.DataFrame(resultados)

df_resultados.to_excel("resultados_whisperx.xlsx", index=False)

print("\n📊 Proceso completado")
print("WER promedio:", df_resultados["wer"].mean())

# PRUEBA DE PIPELINE DE EVALUACIÓN WEV2VEC (clearsrv)

In [ ]:
import os
import time
import torch
import pandas as pd
from transformers import pipeline
from jiwer import wer
import re
import unicodedata
import librosa

# =========================
# NORMALIZACIÓN
# =========================
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = texto.encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r"[^\w\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

# =========================
# CONFIG
# =========================
CARPETA_AUDIO = "llamadas"
EXCEL_PATH = "dataset_llamadas.xlsx"


device = 0 if torch.cuda.is_available() else -1
print("GPU disponible:", torch.cuda.is_available())

# =========================
# CARGAR EXCEL
# =========================
df_ref = pd.read_excel(EXCEL_PATH)

# 🔥 diccionario: audio -> texto real
mapa_textos = dict(zip(df_ref["audio"], df_ref["texto_real"]))


# =========================
# PIPELINE Wav2Vec2 (ESPAÑOL) ✅✅✅
# =========================
asr = pipeline(
    "automatic-speech-recognition",
    model="jonatasgrosman/wav2vec2-large-xlsr-53-spanish",
    device=device,
    chunk_length_s=30,
    stride_length_s=(5, 5)
)

# =========================
# LOOP AUDIOS
# =========================
resultados = []

for archivo in os.listdir(CARPETA_AUDIO):
    
    if not archivo.endswith(".wav"):
        continue
    
    audio_path = os.path.join(CARPETA_AUDIO, archivo)
    audio, sr = librosa.load(audio_path, sr=16000)
    
    print(f"\n🎧 Procesando: {archivo}")

    try:
        start = time.time()

        result = asr(audio)
        texto_pred = result["text"]

        elapsed = time.time() - start

        # 🔥 obtener texto real desde Excel
        texto_real = mapa_textos.get(archivo, None)

        if texto_real:
            wer_score = wer(
                normalizar(texto_real),
                normalizar(texto_pred)
            )
        else:
            wer_score = None

        resultados.append({
            "audio": archivo,
            "texto_real": texto_real,
            "texto_pred": texto_pred,
            "wer": wer_score,
            "tiempo_seg": elapsed
        })

        print(f"WER: {wer_score:.3f}" if wer_score else "Sin referencia")
        print(f"Tiempo: {elapsed:.2f}s")

    except Exception as e:
        print(f"❌ Error en {archivo}: {e}")

# =========================
# RESULTADOS
# =========================
df_resultados = pd.DataFrame(resultados)

df_resultados.to_excel("resultados_wev2vec.xlsx", index=False)

print("\n📊 Proceso completado")